# Multiverse Hybrid v3.0 — Stage 7 Settlement + Frozen A/B/C Evaluation v1

Independent Governance Audit APPROVE 後の DEV2000 専用実行です。**iPhoneでは『ランタイム → すべてのセルを実行』だけ**で構いません。

- Settlement を開く前にコード・Stage2・Prediction・Universe・Provenance を厳密照合
- ECON_HOLDOUT1000 は触らない
- Segment C は Final Freeze 後のみ評価
- 途中停止時は同一Freezeの技術再開のみ。完成済みC台帳があればC再採点なし
- `files.download()` は使わず、ログとReceiptはDriveへ逐次保存


In [ ]:
!pip -q install lxml numpy

from google.colab import drive
from pathlib import Path
import subprocess, shutil, json, hashlib, os, sys, time

drive.mount('/content/drive')
MY=Path('/content/drive/MyDrive')
REPO=Path('/content/multiverse-research-stage7')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])

EXPECTED={
 'v3/historical_all_market/stage7_settlement_bulk_runner_v1.py':'b258ced7250071d642ebf2d40127146b22e9a313',
 'v3/historical_all_market/stage7_eval_core_v1.py':'4787faee7259b0c40733ee9bba84e7909eeae51f',
 'v3/historical_all_market/stage7_historical_evaluation_v1.py':'4d8da6d6fdb38437fcb8d381e6ea3ef2a6592b52',
 'v3/historical_all_market/kdreams_settlement_recovery_v1.py':'b8b8ab0e0904541bd6fc45e7fe415d323e63ec45',
 'v3/historical_all_market/stage456_preoutcome_decision_engine_v1.py':'a0ed6984969b0b98af1b074ef9fd2348f16604a0',
 'v3/historical_all_market/governance/INDEPENDENT_GOVERNANCE_STAGE7_SETTLEMENT_APPROVE_RECEIPT_v1.json':'71e87740ded33ea73c3f534d39830080ad8b43bb',
 'v3/historical_all_market/governance/STAGE3_TICKET_FILTER_FAMILY_PREREG_v1.md':'ba4175bb044bcacfa66a7b8d089e92c04762b2e6',
 'v3/historical_all_market/governance/STAGE4_CONSENSUS_AGREEMENT_GATE_PREREG_v1.md':'f5bb38e97dd2543842308f9b8ee401957d2e5216',
 'v3/historical_all_market/governance/STAGE5_PORTFOLIO_TEMPLATE_PREREG_v1.md':'f13b5aa5584d260d30032c269cfc205a312f2426',
 'v3/historical_all_market/governance/STAGE6_BANKROLL_RISK_POLICY_PREREG_v1.md':'7dc0ac09440755ad1c43959237c0d975be11b245',
 'v3/historical_all_market/governance/STAGE7_TIME_SPLIT_SELECTION_VALIDATION_PREREG_v1.md':'0cb70520777d4ac9d00ddd90b888df1f403c3a7e',
 'v3/historical_all_market/governance/STAGE7_EXECUTION_CONVENTIONS_FREEZE_v1.md':'b388ef5622d4c92ae4df96ad0105882b4994adf4',
}
for rel,exp in EXPECTED.items():
    obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
    if obs!=exp: raise RuntimeError(f'FAIL-CLOSED Git blob mismatch {rel}: {obs} != {exp}')
subprocess.check_call([sys.executable,'-m','py_compile',
    str(REPO/'v3/historical_all_market/stage7_settlement_bulk_runner_v1.py'),
    str(REPO/'v3/historical_all_market/stage7_eval_core_v1.py'),
    str(REPO/'v3/historical_all_market/stage7_historical_evaluation_v1.py')])
print('✅ EXACT CODE / GOVERNANCE BINDINGS + PY_COMPILE PASS')

def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()

U=MY/'MULTIVERSE_DEV2000_UNIVERSE_RECOVERY'/'DEV2000_UNIVERSE_v1.csv'
PROV=MY/'MULTIVERSE_DEV2000_RESULT_COLLECTION_v3_HARDENED'/'DEV2000_RESULT_PROVENANCE_v3.jsonl'
RAW=MY/'MULTIVERSE_DEV2000_RESULT_COLLECTION_v3_HARDENED'/'RAW_RESULT_QUARANTINE'
S2=MY/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1'/'DEV2000_ALL_MARKET_PRICE_EV_CATALOG_v1.jsonl'
PRED=MY/'MULTIVERSE_DEV2000_PREDICTION_LOCK_v3_IPHONE_LITE'/'DEV2000_CANDIDATE_A_B1A_RECONSTITUTED_v1_PREDICTIONS.csv'
required=[U,PROV,S2,PRED]
for p in required:
    if not p.is_file(): raise RuntimeError(f'FAIL-CLOSED missing input: {p}')
if not RAW.is_dir(): raise RuntimeError(f'FAIL-CLOSED missing RAW quarantine: {RAW}')
INPUT_SHA={
 str(U):'eb561c9cad5121cf689b237d44a08d089f375a2b2b728e34e91a48338446f3b1',
 str(PROV):'0e9dbba0bf0427bd1b5903c196a93a31678375170e6d5164b3d8d8f052ca97f1',
 str(S2):'34ad32bed6e8b4d700864c46f4533bef1da254c7d87dc7ffe6ec266fd74530dc',
 str(PRED):'772eca4d26f177b94a86ccf7c1b8486e3cdbac0cae454d76ce91fadeca5f1d51',
}
for ps,exp in INPUT_SHA.items():
    obs=sha256_file(Path(ps)); print('[PREOPEN SHA]',Path(ps).name,obs)
    if obs!=exp: raise RuntimeError(f'FAIL-CLOSED pre-open input SHA mismatch {ps}: {obs} != {exp}')
print('✅ PRE-SETTLEMENT INPUT BINDINGS PASS')

OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE7_SETTLEMENT_EVAL_v1'
OUT.mkdir(parents=True,exist_ok=True)
PRE=OUT/'STAGE7_PREOPEN_RUNTIME_BINDING_v1.json'
PRE.write_text(json.dumps({
 'record':'STAGE7_PREOPEN_RUNTIME_BINDING_v1','status':'PASS_BEFORE_SETTLEMENT_OPEN',
 'audit_snapshot_commit':'a0360b1c5622b0664e8180186a40eca9827fc63e',
 'code_git_blobs':EXPECTED,'input_sha256':INPUT_SHA,'ECON_HOLDOUT1000':'SEALED',
 'stage7_realized_scientific_trial_count_before_open':0
},ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
print('PREOPEN RECEIPT:',PRE)

def run_logged(cmd,log_path):
    log_path=Path(log_path); log_path.parent.mkdir(parents=True,exist_ok=True)
    with log_path.open('a',encoding='utf-8') as lf:
        lf.write('\n=== START '+time.strftime('%Y-%m-%d %H:%M:%S')+' ===\n')
        lf.write('CMD: '+' '.join(map(str,cmd))+'\n'); lf.flush(); os.fsync(lf.fileno())
        p=subprocess.Popen(list(map(str,cmd)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            print(line,end=''); lf.write(line); lf.flush()
        rc=p.wait(); lf.write(f'\nRETURN CODE = {rc}\n'); lf.flush(); os.fsync(lf.fileno())
    return rc

FINAL=OUT/'STAGE7_FINAL_EXECUTION_RECEIPT_v1.json'
already=False
if FINAL.is_file():
    try:
        z=json.loads(FINAL.read_text(encoding='utf-8'))
        already=z.get('status') in {'PASS_COMPLETE','HALT_NO_A_ELIGIBLE_CONFIGURATION','HALT_NO_B_VALIDATED_CONFIGURATION'}
    except Exception: already=False
if already:
    print('✅ STAGE7 ALREADY FINAL — NO RESCORE')
    print(FINAL.read_text(encoding='utf-8'))
else:
    runner=REPO/'v3/historical_all_market/stage7_settlement_bulk_runner_v1.py'
    rc=run_logged([sys.executable,runner,'--mydrive',MY,'--repo-root',REPO],OUT/'STAGE7_SETTLEMENT_BULK_RUN_LOG_v1.txt')
    if rc!=0:
        fatal=OUT/'STAGE7_SETTLEMENT_BULK_FATAL_v1.json'
        if fatal.is_file(): print(fatal.read_text(encoding='utf-8'))
        raise RuntimeError('Stage7 Settlement bulk stopped FAIL-CLOSED; see Drive log/fatal receipt.')
    evaluator=REPO/'v3/historical_all_market/stage7_historical_evaluation_v1.py'
    rc=run_logged([sys.executable,evaluator,'--mydrive',MY,'--repo-root',REPO],OUT/'STAGE7_EVALUATION_RUN_LOG_v1.txt')
    if rc!=0:
        fatal=OUT/'STAGE7_EVALUATION_FATAL_v1.json'
        if fatal.is_file(): print(fatal.read_text(encoding='utf-8'))
        raise RuntimeError('Stage7 evaluation stopped FAIL-CLOSED; see Drive log/fatal receipt.')
    if not FINAL.is_file(): raise RuntimeError('FAIL-CLOSED evaluator returned 0 but final receipt missing')
    print('\n=== STAGE7 FINAL RECEIPT ===')
    print(FINAL.read_text(encoding='utf-8'))
    C=OUT/'STAGE7_SEGMENT_C_OOS_RECEIPT_v1.json'
    if C.is_file():
        print('\n=== SEGMENT C OOS RECEIPT ===')
        print(C.read_text(encoding='utf-8'))
